# I/O

> input/output management

In [1]:
#| default_exp io

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export

from fastai.vision.all import *
from fastai.data.all import *
from torchio import ScalarImage, ToCanonical, Resample
import multipagetiff as mtif

from bioMONAI.core import torchTensor, torch_from_numpy

## Image Readers

### Tiff reader

In [4]:
#| export
def tiff2torch(file_path: str):
    '''
    Load tiff into pytorch tensor
    '''
    import tifffile as tiff
    
    img = np.array(tiff.imread(file_path))
    return torch_from_numpy(img)

In [5]:
#| export

def tiff_reader(path,  # The path to the TIFF file to be read
                units='um',  # The units for the image data.
               ):

    """
    Reads a TIFF file and returns the image data along with an identity affine matrix.

    #### Parameters
    path (str): The path to the TIFF file to be read.
    units (str, optional): The units for the image data. Default is 'um' (micrometers).

    #### Returns
    tuple: A tuple containing:
        - data (numpy.ndarray): The image data read from the TIFF file as a 4D array (1, Z, Y, X).
        - affine (numpy.ndarray): A 4x4 identity affine matrix.
    """

    # Read the TIFF stack using mtif.read_stack with the specified units
    stack = mtif.read_stack(path, units=units)

    # Convert the stack pages to a NumPy array of type float32
    data = stack.pages.astype(np.float32)

    # Add a new axis to the data to make it a 4D array (1, Z, Y, X)
    data = data[None, :, :, :]

    # Create a 4x4 identity affine matrix
    affine = np.eye(4) # to be changed

    # Return the image data and the affine matrix
    return data, affine

In [6]:
file_path = 'data_examples/example_tiff.tiff'
d, _ = tiff_reader(file_path)
d.shape




(1, 96, 512, 512)

### Lif reader

In [7]:
#| export
from aicsimageio import AICSImage

def lif_reader(path, # The path to the LIF file to be read
               units='um', # The units for the image data.
               scene=0, # The scene index to be read from the LIF file
               time=0, # The time index to be read from the LIF file
               reconstruct_mosaic=False, # Whether to reconstruct a mosaic image from the file
              ):

    """
    Reads a LIF (Leica Image File) and returns the image data along with an identity affine matrix.

    Parameters:
    path (str): The path to the LIF file to be read.
    units (str, optional).
    scene (int, optional): The scene index to be read from the LIF file. Default is 0.
    time (int, optional): The time index to be read from the LIF file. Default is 0.
    reconstruct_mosaic (bool, optional): Whether to reconstruct a mosaic image from the file. Default is False.

    Returns:
    tuple: A tuple containing:
        - data (numpy.ndarray): The image data read from the LIF file in ZYX format.
        - affine (numpy.ndarray): A 4x4 identity affine matrix.
    """
    # Create an AICSImage object for the specified file
    imagen_aics = AICSImage(path, reconstruct_mosaic=reconstruct_mosaic)
    # set scence

    # Retrieve the image data in ZYX format for the specified time point
    data = imagen_aics.get_image_data("ZYX", T=time) 
    
    # Create a 4x4 identity affine matrix
    affine = np.eye(4)

    # Return the image data and the affine matrix
    return data, affine

In [8]:
# file_path_2 = 'data_examples/2022-12-05.lif'

# e, _ = lif_reader(file_path_2, units='um')

### CZI reader

In [9]:
#| export
from aicsimageio.readers import CziReader

def czi_reader(path, # The path to the CZI file to be read
              ):

    """
    Reads a CZI (Zeiss CZI format) file and returns the image data along with an identity affine matrix.

    Parameters:
    path (str): The path to the CZI file to be read.

    Returns:
    tuple: A tuple containing:
        - data (numpy.ndarray): The image data read from the CZI file.
        - affine (numpy.ndarray): A 4x4 identity affine matrix.
    """

    # Create a CziReader for the specified file
    reader = CziReader(path)

    # Convert the file to a Numpy Array
    data = reader.data  

    # Create a 4x4 identity affine NumpyArray
    affine = np.eye(4)

    # Return the image data and the affine matrix
    return data, affine

In [10]:
file_path_3 = 'data_examples/example_czi.czi'
f, _ = czi_reader(file_path_3)
f.shape

(1, 1, 1, 1, 1, 1, 1, 1, 1920, 1920)

### Image Reader

In [11]:
#| export

def _image_reader(path, # The file path to the image             
                 ):

    """
    Reads an image from the specified path using AICSImage library.

    Parameters:
        path (str): The file path to the image.

    Returns:
        tuple: A tuple containing the image data and its affine transformation matrix.
               The image data is a NumPy array representing the image.
               The affine transformation matrix is a 4x4 NumPy array.
    """

    # Read image using AICSImage library
    image_aics = AICSImage(path, reconstruct_mosaic=False)
        
    # Support for tiff files    
    path = str(path)
    if (path[-4:]=="tiff"):

        # Reorder for tiff files
        data = image_aics.get_image_data("CZYX", T=0)  # returns 4D CZYX numpy array
        
        affine = np.eye(4) #to change
        
        return data, affine

    if (path[-3:]=="tif"):

        # Reorder for tif files
        data = image_aics.get_image_data("CZYX", T=0)  # returns 4D CZYX numpy array
        
        affine = np.eye(4) #to change
        
        return data, affine
    

    # Convert to numpy array    
    data = image_aics.data

    # Remove singleton dimensions
    data = np.squeeze(data)
    
    # Create an identity affine transformation matrix
    affine = np.eye(4)

    # Return the image data and the affine matrix
    return data, affine


In [12]:
file_path = 'data_examples/example_tiff.tiff'
test_img, _ = _image_reader(file_path)
test_img.shape

(1, 96, 512, 512)

### H5 Reader

In [13]:
#| export
import h5py

def h5_reader(path, dataset):
    with h5py.File(path, 'r') as hdf:
            ls = list(hdf.keys())
            print('List of datasets in this file: \n',ls)
            data = hdf.get(dataset)
            dataset1 = np.array(data)
            print('Shape of dataset1: \n', dataset1.shape)

    return dataset1


In [14]:
# f = h5_reader('data_examples/hdf5_data.h5','dataset1')


### Preprocessing

In [15]:
#| export

def _preprocess(obj, # The object to preprocess
                reorder, # Whether to reorder the object
                resample # Whether to resample the object
               ):
    """
    Preprocesses the given object.

    Args:
        obj: The object to preprocess.
        reorder: Whether to reorder the object.
        resample: Whether to resample the object.

    Returns:
        The preprocessed object and its original size.
    """
    if reorder:
        transform = ToCanonical()
        obj = transform(obj)
    


    original_size = obj.shape[1:]

    if resample and not all(np.isclose(obj.spacing, resample)):
        transform = Resample(resample)
        obj = transform(obj)

    # if MedBase.affine_matrix is None:
    #     MedBase.affine_matrix = obj.affine

    return obj, original_size


### Load and preprocess

In [16]:
#| export

def _load_and_preprocess(file_path, # Image file path
                         reorder=False, # Whether to reorder data for canonical (RAS+) orientation
                         resample=False, # Whether to resample image to different voxel sizes and dimensions
                         reader=_image_reader # Whether to resample image to different voxel sizes and dimensions
                        ):
    """
    Helper function to load and preprocess an image.

    Args:
        file_path: Image file path.
        reorder: Whether to reorder data for canonical (RAS+) orientation.
        resample: Whether to resample image to different voxel sizes and dimensions.
        dtype: Desired datatype for output.

    Returns:
        tuple: Original image, preprocessed image, and its original size.
    """
    org_img = ScalarImage(file_path, reader=reader)
    input_img, org_size = _preprocess(org_img, reorder, resample)
    
    return org_img, input_img, org_size


In [17]:
# load and process any file
# org_img, input_img, org_size = _load_and_preprocess(file_path)


### Read multichannel data

In [18]:
#| export

def _multi_channel(image_paths: (L, list), # List of image paths (e.g., T1, T2, T1CE, DWI)
                   reorder: bool = False, # Whether to reorder data for canonical (RAS+) orientation
                   resample: list = None, # Whether to resample image to different voxel sizes and dimensions
                   dtype=torchTensor, # Desired datatype for output
                   only_tensor: bool = True, # Whether to return only image tensor
                   squeeze: bool = False # 
                  ):
    """
    Load and preprocess multisequence data.

    Args:
        image_paths: List of image paths (e.g., T1, T2, T1CE, DWI).
        reorder: Whether to reorder data for canonical (RAS+) orientation.
        resample: Whether to resample image to different voxel sizes and dimensions.
        dtype: Desired datatype for output.
        only_tensor: Whether to return only image tensor.
        squeeze: Whether to squeeze or not the image

    Returns:
        torchTensor: A stacked 4D tensor, if `only_tensor` is True.
        tuple: Original image, preprocessed image, original size, if `only_tensor` is False.
    """
    image_data = [_load_and_preprocess(image, reorder, resample) for image in image_paths]
    org_img, input_img, org_size = image_data[-1]

    tensor = torch.stack([img.data[0] for _, img, _ in image_data], dim=0)

    if only_tensor:     
        if squeeze:
            return torch.squeeze(dtype(tensor))
        return dtype(tensor) 

    input_img.set_data(tensor)
    return org_img, input_img, org_size

In [19]:
# file_names = get_image_files('data_examples')

# _multi_channel(file_names);


### Image reader

In [20]:
#| export

def img_reader(file_path: (str, Path, L, list), # Path to the image
               dtype=torchTensor, # Datatype for the return value. Defaults to torchTensor
               reorder: bool = False, # Whether to reorder to canonical orientation
               resample: list = None, # Whether to resample image to different voxel sizes and image dimensions
               only_tensor: bool = True, # To return only an image tensor
              ):
    """Loads and preprocesses a medical image.

    Args:
        file_path: Path to the image. Can be a string, Path object or a list.
        dtype: Datatype for the return value. Defaults to torchTensor.
        reorder: Whether to reorder the data to be closest to canonical 
            (RAS+) orientation. Defaults to False.
        resample: Whether to resample image to different voxel sizes and 
            image dimensions. Defaults to None.
        only_tensor: To return only an image tensor. Defaults to True.

    Returns:
        The preprocessed image. Returns only the image tensor if 
        only_tensor is True, otherwise returns original image, 
        preprocessed image, and original size.
    """
    # if isinstance(file_path, str) and ';' in file_path:
    #     return _multi_channel(
    #         file_path.split(';'), reorder, resample, dtype, only_tensor)
    
    if isinstance(file_path, (L, list)):
        return _multi_channel(file_path, reorder, resample, dtype, only_tensor)

    org_img, input_img, org_size = _load_and_preprocess(
        file_path, reorder, resample)

    if only_tensor:
        return dtype(input_img.data.type(torch.float))

    return org_img, input_img, org_size


In [21]:
#img_reader(file_path_2)

In [22]:
#| hide
import nbdev; nbdev.nbdev_export()